# CartLens — E-commerce Data Exploration & Business Analysis

**CartLens** is an e-commerce analytics project focused on understanding business performance, customer behavior, product profitability, delivery experience, seller performance, payment behavior, and geographic trends.

The analysis uses the **Brazilian E-Commerce Public Dataset by Olist**, which contains information about orders, customers, products, sellers, payments, reviews, and related transactions.

The objective of this exploratory analysis is to transform raw transactional data into meaningful business insights that can support data-driven decision-making.


## Dataset Overview

The **Olist Brazilian E-Commerce Public Dataset** contains transactional and operational data from an e-commerce marketplace in Brazil.

The dataset provides information across multiple business areas, including customers, orders, products, sellers, payments, reviews, and geographic information.

In [1]:
import sys
print(sys.executable)

d:\CartLens_Ecommerce_Analytics\.venv\Scripts\python.exe


### Import necessary libraries

In [2]:
import pandas as pd
import numpy as np


### Load  all datasets all at once

In [3]:
customer_dataset = '../data/raw/olist_customers_dataset.csv'
geoocation_dataset = '../data/raw/olist_geolocation_dataset.csv'
order_item_dataset = '../data/raw/olist_order_items_dataset.csv'
order_payments_dataset = '../data/raw/olist_order_payments_dataset.csv'
order_reviews_dataset = '../data/raw/olist_order_reviews_dataset.csv'
order_dataset = '../data/raw/olist_orders_dataset.csv'
product_dataset = '../data/raw/olist_products_dataset.csv'
seller_dataset = '../data/raw/olist_sellers_dataset.csv'
category_name_translation_dataset = '../data/raw/product_category_name_translation.csv'

df_c = pd.read_csv(customer_dataset)
df_g = pd.read_csv(geoocation_dataset)
df_oi = pd.read_csv(order_item_dataset)
df_op = pd.read_csv(order_payments_dataset)
df_or = pd.read_csv(order_reviews_dataset)
df_o = pd.read_csv(order_dataset)
df_p = pd.read_csv(product_dataset)
df_s = pd.read_csv(seller_dataset)
df_cnt = pd.read_csv(category_name_translation_dataset)


In [4]:
print(df_c.shape)
print(df_o.shape)
print(df_oi.shape)
print(df_op.shape)
print(df_or.shape)
print(df_p.shape)
print(df_s.shape)
print(df_g.shape)
print(df_cnt.shape)

(99441, 5)
(99441, 8)
(112650, 7)
(103886, 5)
(99224, 7)
(32951, 9)
(3095, 4)
(1000163, 5)
(71, 2)


In [5]:
print(df_o.shape)
df_o.count()
df_cnt.head()

(99441, 8)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


### Exploring order dataframe

In [6]:
df_o.head()
df_o.info()
df_o.describe()
df_o.shape


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


(99441, 8)

#### We need to check whether missing values are normal or whether they indicate a data-quality issue.

In [7]:
print(df_o.isnull().sum().idxmax())
print(df_o.isnull().sum().max())


order_delivered_customer_date
2965


##### There are 2,965 missing delivery dates. Many of these are expected because the orders were not delivered. We should not delete them before checking the order status.

#### Why are these delivery dates missing, and should we remove those rows?

In [8]:
df_o[df_o['order_delivered_customer_date'].isnull()]['order_status'].value_counts()


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

##### Only 8 delivered orders have a missing order_delivered_customer_date.

In [9]:
condition = (df_o['order_status'] =='delivered') & (df_o['order_delivered_customer_date'].isnull())
columns = [
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

df_o.loc[condition, columns]


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaN,2018-07-19 00:00:00


##### The dataset marks these orders as delivered, but the actual delivery date is missing.


#### Before using the data for analysis, we need to make sure the order IDs are not duplicated.

In [10]:
df_o['order_id'].duplicated().value_counts()
df_o['order_id'].isnull().sum()


np.int64(0)

##### No duplicate values


In [11]:
print("Missing customer_id: ",df_o['customer_id'].isnull().sum())
print("Duplicate customer_id: ",df_o['customer_id'].duplicated().value_counts())


Missing customer_id:  0
Duplicate customer_id:  customer_id
False    99441
Name: count, dtype: int64


### Analyzing customer dataset

In [12]:
print("unique ",df_c['customer_unique_id'].nunique())
print("customer_id ",df_c['customer_id'].count())

unique  96096
customer_id  99441


In [13]:
# multiple customer_id belong to same customer_unique_id?
print("no of customer_id under same customer_unique_id ")
counts = df_c.groupby('customer_unique_id')['customer_id'].nunique()
print(counts[counts>1])
print((counts > 1).sum())

no of customer_id under same customer_unique_id 
customer_unique_id
00172711b30d52eea8b313a7f2cced02    2
004288347e5e88a27ded2bb23747066c    2
004b45ec5c64187465168251cd1c9c2f    2
0058f300f57d7b93c477a131a59b36c3    2
00a39521eb40f7012db50455bf083460    2
                                   ..
ff36be26206fffe1eb37afd54c70e18b    3
ff44401d0d8f5b9c54a47374eb48c1b8    2
ff8892f7c26aa0446da53d01b18df463    2
ff922bdd6bafcdf99cb90d7f39cea5b3    3
ffe254cc039740e17dd15a5305035928    2
Name: customer_id, Length: 2997, dtype: int64
2997


In [14]:
print("duplicate ",df_c['customer_id'].duplicated().value_counts())

duplicate  customer_id
False    99441
Name: count, dtype: int64


In [15]:
# missing columns
print("missing ",df_c.isnull().sum())
df_c["customer_id"].isin(df_c["customer_id"]).value_counts()

missing  customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


customer_id
True    99441
Name: count, dtype: int64

### Order Items datasets

In [16]:
print("shape: ", df_oi.shape)
df_oi.head()


shape:  (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### Sales analysis

In [17]:
# total orders
total_orders = df_o['order_id'].count()
delivered_orders = df_o.loc[
    df_o['order_status'] == 'delivered', 'order_id'
].count()
print(total_orders)
print(delivered_orders)


99441
96478


In [18]:
# revenue
delivered = df_o[df_o['order_status'] == 'delivered']
sales = delivered.merge(
    df_oi,
    on='order_id',
    how='inner'
)
revenue = sales['price'].sum()
print("revenue: ", revenue)


revenue:  13221498.110000001


In [19]:
# monthly revenue
sales['order_purchase_timestamp'] = pd.to_datetime(
    sales['order_purchase_timestamp']
)
sales['year'] = sales['order_purchase_timestamp'].dt.year
sales['month'] = sales['order_purchase_timestamp'].dt.month

monthly_sales = (
    sales
    .groupby(['year', 'month'])
    .agg(
        revenue = ('price', 'sum'),
        orders = ('order_id', 'nunique')
    )
    .reset_index()
)

monthly_sales = monthly_sales.sort_values(['year', 'month'])
monthly_sales


,year,month,revenue,orders
0,2016,9,134.97,1
1,2016,10,40325.11,265
2,2016,12,10.90,1
3,2017,1,111798.36,750
4,2017,2,234223.40,1653
5,2017,3,359198.85,2546
6,2017,4,340669.68,2303
7,2017,5,489338.25,3546
8,2017,6,421923.37,3135
9,2017,7,481604.52,3872


In [20]:
# monthly AOV
# AOV = Revenue / Number of Orders
monthly_sales['AOV'] = monthly_sales['revenue']/monthly_sales['orders']
monthly_sales.head()


,year,month,revenue,orders,AOV
0,2016,9,134.97,1,134.970000
1,2016,10,40325.11,265,152.170226
2,2016,12,10.90,1,10.900000
3,2017,1,111798.36,750,149.064480
4,2017,2,234223.40,1653,141.695947


In [21]:
# month-over-month revenue growth
monthly_sales['previous_revenue'] = monthly_sales['revenue'].shift(1)
monthly_sales['mom_growth_pct'] = (
    (monthly_sales['revenue'] - monthly_sales['previous_revenue'])/monthly_sales['previous_revenue'] * 100
)
monthly_sales.head()
print("highest revenue")
print(monthly_sales.loc[monthly_sales['revenue'].idxmax()])

print("lowest revenue")
print(monthly_sales.loc[monthly_sales['revenue'].idxmin()])

print("highest AOV")
print(monthly_sales.loc[monthly_sales['AOV'].idxmax()])

print("highest month-over-month growth")
print(monthly_sales.loc[monthly_sales['mom_growth_pct'].idxmax()])


highest revenue
year                  2017.000000
month                   11.000000
revenue             987765.370000
orders                7289.000000
AOV                    135.514525
previous_revenue    648247.650000
mom_growth_pct          52.374694
Name: 13, dtype: float64
lowest revenue
year                 2016.00000
month                  12.00000
revenue                10.90000
orders                  1.00000
AOV                    10.90000
previous_revenue    40325.11000
mom_growth_pct        -99.97297
Name: 2, dtype: float64
highest AOV
year                 2016.000000
month                  10.000000
revenue             40325.110000
orders                265.000000
AOV                   152.170226
previous_revenue      134.970000
mom_growth_pct      29777.091205
Name: 1, dtype: float64
highest month-over-month growth
year                2.017000e+03
month               1.000000e+00
revenue             1.117984e+05
orders              7.500000e+02
AOV                 1.49064

##### Which month had the highest revenue? => 2017
##### Which month had the lowest revenue? => 2016
##### Which month had the highest AOV? =>2016
##### Which month had the largest positive MoM growth? => 2017

#### The dataset contains a small amount of 2016 data, so we need to identify the period that gives us a reliable view of sales performance.

In [22]:
# Identifying the Reliable Sales Period
monthly_sales.sort_values(
    ['year', 'month']
)[['year', 'month', 'orders', 'revenue']]

print(monthly_sales[
    ['year', 'month', 'orders', 'revenue']
].sort_values(['year', 'month']).head())

print(monthly_sales[
    ['year', 'month', 'orders', 'revenue']
].sort_values(['year', 'month']).tail())


   year  month  orders    revenue
0  2016      9       1     134.97
1  2016     10     265   40325.11
2  2016     12       1      10.90
3  2017      1     750  111798.36
4  2017      2    1653  234223.40
    year  month  orders    revenue
18  2018      4    6798  973534.09
19  2018      5    6749  977544.69
20  2018      6    6099  856077.86
21  2018      7    6159  867953.46
22  2018      8    6351  838576.64


##### The important thing is that the dataset starts partway through 2016, so the first months may not represent complete calendar months.

In [23]:
# Creating a reliable analysis dataset
# Exclude the initial partial period and analyze complete/high-volume calendar months from 2017 onward
monthly_sales_analysis = monthly_sales[
    monthly_sales['year'] >= 2017
].copy()

monthly_sales_analysis = monthly_sales_analysis.sort_values(
    ['year', 'month']
).reset_index(drop=True)
monthly_sales_analysis.head()


,year,month,revenue,orders,AOV,previous_revenue,mom_growth_pct
0,2017,1,111798.36,750,149.064480,10.90,1.025573e+06
1,2017,2,234223.40,1653,141.695947,111798.36,1.095052e+02
2,2017,3,359198.85,2546,141.083602,234223.40,5.335737e+01
3,2017,4,340669.68,2303,147.924307,359198.85,-5.158471e+00
4,2017,5,489338.25,3546,137.997250,340669.68,4.364009e+01


In [24]:
# highest revenue after excluding unreliable period
highest_revenue_month = monthly_sales_analysis.loc[
    monthly_sales_analysis['revenue'].idxmax()
]
highest_revenue_month[['year', 'month', 'orders', 'revenue', 'AOV']]


year         2017.000000
month          11.000000
orders       7289.000000
revenue    987765.370000
AOV           135.514525
Name: 10, dtype: float64

In [25]:
# highest AOV
highest_AOV_month = monthly_sales_analysis.loc[
    monthly_sales_analysis['AOV'].idxmax()
]
highest_AOV_month[['year', 'month', 'orders', 'revenue', 'AOV']]


year         2017.00000
month           1.00000
orders        750.00000
revenue    111798.36000
AOV           149.06448
Name: 0, dtype: float64

In [26]:
# meaningful MoM growth
mom_analysis = monthly_sales_analysis[
    ~(
        (monthly_sales_analysis['year'] == 2017) &
        (monthly_sales_analysis['month'] == 1)
    )
].copy()
highest_growth_month = mom_analysis.loc[
    mom_analysis['mom_growth_pct'].idxmax()
]
highest_growth_month[['year', 'month', 'orders', 'revenue', 'previous_revenue', 'mom_growth_pct']]


print("Highest Revenue Month:")
print(highest_revenue_month)

print("\nHighest AOV Month:")
print(highest_AOV_month)

print("\nHighest Meaningful MoM Growth:")
print(highest_growth_month)


Highest Revenue Month:
year                  2017.000000
month                   11.000000
revenue             987765.370000
orders                7289.000000
AOV                    135.514525
previous_revenue    648247.650000
mom_growth_pct          52.374694
Name: 10, dtype: float64

Highest AOV Month:
year                2.017000e+03
month               1.000000e+00
revenue             1.117984e+05
orders              7.500000e+02
AOV                 1.490645e+02
previous_revenue    1.090000e+01
mom_growth_pct      1.025573e+06
Name: 0, dtype: float64

Highest Meaningful MoM Growth:
year                  2017.000000
month                    2.000000
revenue             234223.400000
orders                1653.000000
AOV                    141.695947
previous_revenue    111798.360000
mom_growth_pct         109.505220
Name: 1, dtype: float64


### Product & Category


#### Business Problem
We want to understand which products and categories are driving sales.


##### Which product categories generate the most revenue?
##### Which categories generate the most orders?
##### Which products are the top sellers?
##### Which categories have high revenue but relatively low order volume?
##### Which categories have high order volume but relatively low revenue?
##### Does high order volume actually mean high revenue?

In [27]:
print(df_p.head())

print(df_cnt.head())

delivered = df_o[
    (df_o['order_status'] == 'delivered') &
    (pd.to_datetime(df_o['order_purchase_timestamp']).dt.year >= 2017)
]

sales = delivered.merge(
    df_oi,
    on='order_id',
    how='inner'
)

product_wise_sales = sales.merge(
    df_p,
    on='product_id',
    how='left'
)

product_wise_sales.columns


                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   
3                 27.0                       261.0                 1.0   
4                 37.0                       402.0                 4.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0       

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm'],
      dtype='str')

In [28]:
# revenue by category
category_wise_revenue = (
    product_wise_sales
    .groupby('product_category_name')['price']
    .sum()
    .sort_values(ascending=False)
)
print("Revenue by category")
print(category_wise_revenue)


Revenue by category
product_category_name
beleza_saude                     1229557.50
relogios_presentes               1163465.91
cama_mesa_banho                  1022955.77
esporte_lazer                     952840.40
informatica_acessorios            888055.59
                                    ...    
flores                              1110.04
casa_conforto_2                      760.27
cds_dvds_musicais                    730.00
fashion_roupa_infanto_juvenil        519.95
seguros_e_servicos                   283.29
Name: price, Length: 73, dtype: float64


In [29]:
# orders by category
category_wise_sales = (
    product_wise_sales
    .groupby('product_category_name')['order_id']
    .nunique()
    .sort_values(ascending=False)
)
print("Orders by category")
print(category_wise_sales)


Orders by category
product_category_name
cama_mesa_banho                                  9267
beleza_saude                                     8610
esporte_lazer                                    7513
informatica_acessorios                           6518
moveis_decoracao                                 6258
                                                 ... 
portateis_cozinha_e_preparadores_de_alimentos      13
cds_dvds_musicais                                  12
fashion_roupa_infanto_juvenil                       7
pc_gamer                                            7
seguros_e_servicos                                  2
Name: order_id, Length: 73, dtype: int64


In [30]:
# category AOV
category_analysis = (
    product_wise_sales
    .groupby('product_category_name')
    .agg(
        revenue=('price', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)
category_analysis['AOV'] = (
    category_analysis['revenue'] / category_analysis['orders']
)
category_analysis.sort_values(
    'AOV',
    ascending=False
).head(10)


,product_category_name,revenue,orders,AOV
61,pcs,218684.14,177,1235.503616
64,portateis_casa_forno_e_cafe,46589.56,72,647.077222
29,eletrodomesticos_2,107953.95,227,475.568062
0,agro_industria_e_comercio,70566.10,177,398.678531
65,portateis_cozinha_e_preparadores_de_alimentos,3933.63,13,302.586923
45,instrumentos_musicais,184315.74,611,301.662422
31,eletroportateis,182754.12,609,300.088867
71,telefonia_fixa,54610.33,207,263.818019
25,construcao_ferramentas_seguranca,38773.22,159,243.856730
19,climatizacao,51974.27,242,214.769711


##### beleza_saude leads through high order volume, while relogios_presentes generates similar revenue with substantially higher value per order.

In [31]:
# top 10 products by revenue
product_wise_revenue = (
    product_wise_sales
    .groupby(['product_id', 'product_category_name'])
    .agg(
        revenue = ('price', 'sum')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
    .head(10)
)
print(product_wise_revenue)


                             product_id   product_category_name   revenue
23034  bb50f2e236e5eea0100680137654686c            beleza_saude  63560.00
13446  6cdd53843498f92890544667809f1595            beleza_saude  53652.30
26405  d6160fb7873f184099d9bc95e30376af                     pcs  45949.35
25853  d1c427060a0f73f6b889a5c7c61f2ac4  informatica_acessorios  45620.56
18877  99a4788cb24856965c36a24e339b6058         cama_mesa_banho  42049.66
7713   3dd2a17168ec895c781a9191c1e95ad7  informatica_acessorios  40782.80
4789   25c38557cf793876c5abdd5931f922db                   bebes  38907.32
11812  5f504b3a1c75b73d6151be81eb05bdc9              cool_stuff  37733.90
10382  53b36df67ebb7c41585e8d54d6772e08      relogios_presentes  37454.63
21159  aca2eb7d00ea1a7b8ebd4e68314663af        moveis_decoracao  37104.30


In [32]:
# top 10 products by order count
product_wise_order = (
    product_wise_sales
    .groupby(['product_id', 'product_category_name'])
    .agg(
        orders = ('order_id', 'nunique')
    )
    .sort_values('orders', ascending=False)
    .reset_index()
    .head()
)
print(product_wise_order)


                         product_id   product_category_name  orders
0  99a4788cb24856965c36a24e339b6058         cama_mesa_banho     456
1  aca2eb7d00ea1a7b8ebd4e68314663af        moveis_decoracao     425
2  422879e10f46682990de24d770e7f83d      ferramentas_jardim     352
3  d1c427060a0f73f6b889a5c7c61f2ac4  informatica_acessorios     313
4  389d119b48cf3043d311335e499d9c6b      ferramentas_jardim     309


In [33]:
# high volume vs high value categories
category_analysis = category_analysis.sort_values(
    'revenue',
    ascending=False
)
category_analysis.head(10)


,product_category_name,revenue,orders,AOV
11,beleza_saude,1229557.50,8610,142.805749
66,relogios_presentes,1163465.91,5491,211.885979
13,cama_mesa_banho,1022955.77,9267,110.386940
32,esporte_lazer,952840.40,7513,126.825556
44,informatica_acessorios,888055.59,6518,136.246639
54,moveis_decoracao,706237.17,6258,112.853495
72,utilidades_domesticas,614341.62,5734,107.140150
26,cool_stuff,609158.00,3552,171.497185
8,automotivo,577838.39,3802,151.982743
40,ferramentas_jardim,469135.40,3443,136.257740


High volume + high value → strong performers

High volume + low value → many orders but relatively small orders

Low volume + high value → fewer orders but expensive purchases

Low volume + low value → weak performers

Revenue alone does not tell us whether a category is financially attractive. We also want to compare estimated product cost and freight.
#### Which products and categories are actually financially attractive after considering product cost and freight?

### Setting a Reference Selling Price
##### Important Note
Olist does not provide actual product cost/COGS. The product costs used here are **synthetic estimates** created for analytical demonstration, so the profit and margin figures should not be treated as actual company profit.

In [34]:
product_reference_price = (
    product_wise_sales
    .groupby('product_id')
    .agg(
        average_selling_price=('price', 'mean')
    )
    .reset_index()
)
print(product_reference_price.head())

print(product_reference_price.shape)
print(product_reference_price.nunique())
print(product_wise_sales['price'].isnull().sum())
print(product_wise_sales['product_id'].isnull().sum())

product_reference_price[
    product_reference_price['average_selling_price'].isnull()
].head()
product_reference_price['average_selling_price'].describe()
product_reference_price.sort_values(
    'average_selling_price',
    ascending=False
).head(10)
conditions = [
    product_reference_price['average_selling_price'] <= 50,
    product_reference_price['average_selling_price'] <= 150,
    product_reference_price['average_selling_price'] <= 500,
    product_reference_price['average_selling_price'] > 500
]
choices = [
    0.60,
    0.65,
    0.70,
    0.75
]
product_reference_price['estimated_cost_pct'] = np.select(
    conditions, choices
)

product_reference_price['estimated_product_cost'] = (
    product_reference_price['average_selling_price'] * product_reference_price['estimated_cost_pct']
)

product_reference_price


                         product_id  average_selling_price
0  00066f42aeeb9f3007548bb9d3f33c38                 101.65
1  00088930e925c41fd95ebfe695fd2655                 129.90
2  0009406fd7479715e4bef61dd91f2462                 229.00
3  000b8f95fcb9e0096488278317764d19                  58.90
4  000d9be29b5207b54e86aa1b1ac54872                 199.00
(32081, 2)
product_id               32081
average_selling_price     8621
dtype: int64
0
0


,product_id,average_selling_price,estimated_cost_pct,estimated_product_cost
0,00066f42aeeb9f3007548bb9d3f33c38,101.65,0.65,66.0725
1,00088930e925c41fd95ebfe695fd2655,129.90,0.65,84.4350
2,0009406fd7479715e4bef61dd91f2462,229.00,0.70,160.3000
3,000b8f95fcb9e0096488278317764d19,58.90,0.65,38.2850
4,000d9be29b5207b54e86aa1b1ac54872,199.00,0.70,139.3000
...,...,...,...,...
32076,fff6177642830a9a94a0f2cba5e476d1,114.99,0.65,74.7435
32077,fff81cc3158d2725c0655ab9ba0f712c,90.00,0.65,58.5000
32078,fff9553ac224cec9d15d49f5a263411f,32.00,0.60,19.2000
32079,fffdb2d0ec8d6a61f0a0a0db3f25b441,33.99,0.60,20.3940


##### Product costs are synthetic estimates created for analytical demonstration because the original Olist dataset does not provide actual cost-of-goods-sold data.

In [35]:
# creating supplementary dataset
product_costs = product_reference_price[
    ['product_id', 'estimated_product_cost']
].copy()
product_costs = product_costs.rename(
    columns={'estimated_product_cost': 'product_cost'}
)
print(product_costs.head())
print(product_costs.shape)
gross_profit_per_product = product_wise_sales.merge(
    product_costs,
    on='product_id',
    how='inner'
)
gross_profit_per_product.head()
gross_profit_per_product['gross_profit'] = (
    gross_profit_per_product['price']
    - gross_profit_per_product['product_cost']
)
gross_profit_per_product[
    ['product_id', 'price', 'product_cost', 'gross_profit']
].head(10)

(gross_profit_per_product['gross_profit'] < 0).sum()
product_profitability = (
    gross_profit_per_product
    .groupby('product_id')
    .agg(
        revenue=('price', 'sum'),
        product_cost=('product_cost', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)

product_profitability['gross_margin_pct'] = (
    product_profitability['gross_profit']
    / product_profitability['revenue']
    * 100
)

product_profitability[
    ['product_id', 'revenue', 'product_cost',
     'gross_profit', 'orders', 'gross_margin_pct']
].head(10)

(product_profitability['gross_profit'] < 0).sum()


                         product_id  product_cost
0  00066f42aeeb9f3007548bb9d3f33c38       66.0725
1  00088930e925c41fd95ebfe695fd2655       84.4350
2  0009406fd7479715e4bef61dd91f2462      160.3000
3  000b8f95fcb9e0096488278317764d19       38.2850
4  000d9be29b5207b54e86aa1b1ac54872      139.3000
(32081, 2)


np.int64(0)

In [36]:
# Profit after freight
gross_profit_per_product['profit_after_freight'] = (
    gross_profit_per_product['gross_profit']
    - gross_profit_per_product['freight_value']
)
gross_profit_per_product[
    [
        'product_id',
        'price',
        'product_cost',
        'freight_value',
        'gross_profit',
        'profit_after_freight'
    ]
].head(10)

(
    gross_profit_per_product['profit_after_freight'] < 0
).sum()


np.int64(30648)

In [37]:
# category profitability
category_profitability = (
    gross_profit_per_product
    .groupby('product_category_name')
    .agg(
        revenue = ('price', 'sum'),
        product_cost = ('product_cost', 'sum'),
        gross_profit = ('gross_profit', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)

category_profitability['gross_margin_pct'] = (
    category_profitability['gross_profit']
    / category_profitability['revenue']
    * 100
)

category_profitability.sort_values(
    'gross_profit',
    ascending=False
).head(10)

category_profitability.sort_values(
    'gross_margin_pct',
    ascending=False
).head(10)


,product_category_name,revenue,product_cost,gross_profit,orders,gross_margin_pct
42,fraldas_higiene,1365.89,835.5285,530.3615,24,38.829005
41,flores,1110.04,692.0190,418.0210,29,37.658192
36,fashion_roupa_feminina,2580.04,1647.0040,933.0360,35,36.163625
37,fashion_roupa_infanto_juvenil,519.95,331.9690,187.9810,7,36.153669
15,casa_conforto_2,760.27,486.6555,273.6145,24,35.989122
1,alimentos,28651.25,18425.2175,10226.0325,440,35.691401
10,bebidas,21529.84,13924.6230,7605.2170,287,35.324076
17,cds_dvds_musicais,730.00,474.5000,255.5000,12,35.000000
6,artigos_de_natal,8737.84,5686.6190,3051.2210,125,34.919625
2,alimentos_bebidas,14942.88,9728.8745,5214.0055,221,34.892909


##### Some categories have higher estimated margins, including `fraldas_higiene`, `flores`, and `fashion_roupa_feminina`.

##### Business Insight
A high margin does not automatically mean a category is more important. For example, `fraldas_higiene` has a high estimated margin but very little revenue. So we should consider both **margin and business scale**.

### Revenue vs. Margin

In [38]:
high_revenue_low_margin = category_profitability[
    (category_profitability['revenue'] >= 500000) &
    (category_profitability['gross_margin_pct'] < 32)
]

high_revenue_low_margin.sort_values(
    'revenue',
    ascending=False
)


,product_category_name,revenue,product_cost,gross_profit,orders,gross_margin_pct
11,beleza_saude,1229557.50,847277.6445,382279.8555,8610,31.090848
66,relogios_presentes,1163465.91,822193.2775,341272.6325,5491,29.332414
26,cool_stuff,609158.00,422611.0490,186546.9510,3552,30.623738
8,automotivo,577838.39,399995.7870,177842.6030,3802,30.777222


##### relogios_presentes generates high revenue but has a comparatively lower estimated gross margin, suggesting an opportunity to investigate pricing and product-cost structure


In [39]:
# overall profitability
total_revenue = category_profitability['revenue'].sum()
total_product_cost = category_profitability['product_cost'].sum()
total_gross_profit = category_profitability['gross_profit'].sum()

overall_gross_margin = (
    total_gross_profit / total_revenue
) * 100

print("Total Revenue:", total_revenue)
print("Total Product Cost:", total_product_cost)
print("Total Gross Profit:", total_gross_profit)
print("Overall Gross Margin:", overall_gross_margin)


Total Revenue: 13010366.389999999
Total Product Cost: 8909844.459499998
Total Gross Profit: 4100521.9305
Overall Gross Margin: 31.517343997719653


### Final Profitability: Freight Impact
##### After accounting for freight charges, how much of the estimated gross profit remains?

In [40]:
gross_profit_per_product['profit_after_freight'] = (
    gross_profit_per_product['gross_profit']
    - gross_profit_per_product['freight_value']
)
category_freight_analysis = (
    gross_profit_per_product
    .groupby('product_category_name')
    .agg(
        revenue=('price', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        freight=('freight_value', 'sum'),
        contribution_after_freight=('profit_after_freight', 'sum')
    )
    .reset_index()
)

category_freight_analysis['contribution_margin_pct'] = (
    category_freight_analysis['contribution_after_freight']
    / category_freight_analysis['revenue']
) * 100

category_freight_analysis.sort_values(
    'contribution_after_freight'
).head(10)
gross_profit_per_product['profit_after_freight'] = (
    gross_profit_per_product['gross_profit']
    - gross_profit_per_product['freight_value']
)
print(
    gross_profit_per_product[
        ['price', 'product_cost', 'freight_value', 'gross_profit', 'profit_after_freight']
    ].head(10)
)
print(
    gross_profit_per_product[
        ['gross_profit', 'freight_value', 'profit_after_freight']
    ].describe()
)


    price  product_cost  freight_value  gross_profit  profit_after_freight
0   29.99     17.994000           8.72     11.996000              3.276000
1  118.70     77.506250          22.76     41.193750             18.433750
2  159.90    111.463333          19.22     48.436667             29.216667
3   45.00     25.650000          27.20     19.350000             -7.850000
4   19.90     13.080000           8.72      6.820000             -1.900000
5  147.90     95.395000          27.36     52.505000             25.145000
6   59.99     24.535245          15.17     35.454755             20.284755
7   19.90     11.940000          16.05      7.960000             -8.090000
8  149.99    112.854875          19.77     37.135125             17.365125
9   99.00     61.657227          30.53     37.342773              6.812773
        gross_profit  freight_value  profit_after_freight
count  109880.000000  109880.000000         109880.000000
mean       37.817482      19.949881             17.867601
s

#### We want to understand how valuable our customers are and how often they come back.

1. Who are our most valuable customers?
2. How many customers purchase only once?
3. How many customers come back?
4. Which customer groups should we focus on?


#### Customer Lifetime Value

In [41]:

customer_sales = (
    delivered
    .merge(
        df_c,
        on='customer_id',
        how='left'
    )
    .merge(
        df_oi,
        on='order_id',
        how='inner'
    )
)

customer_sales.head()
print(customer_sales.columns)
print(customer_sales.shape)
customer_sales[
    ['customer_unique_id', 'order_id', 'price']
].head(10)


Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
(109880, 18)


,customer_unique_id,order_id,price
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,29.99
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,118.70
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,159.90
3,7c142cf63193a1473d2e66489a9ae977,949d5b44dbf5de918fe9c16f97b45f8a,45.00
4,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,19.90
5,80bb27c7c16e8f973207a5086ab329e2,a4591c265e18cb1dcee52889e2d8acc3,147.90
6,932afa1e708222e5821dac9cd5db4cae,6514b8ad8028c9f2cc2374ded245783f,59.99
7,39382392765b6dc74812866ee5ee92a7,76c6e866289321a7c93b82b54852dc33,19.90
8,299905e3934e9e181bfb2e164dd4b4f8,e69bfb5eb88e0ed6a785585b27e16dbf,149.99
9,f2a85dec752b8517b5e58a06ff3cd937,e6ce16cb79ec1d90b1da9085a6118aeb,99.00


In [42]:
# each customer's total revenue
customer_revenue = (
    customer_sales
    .groupby('customer_unique_id')
    .agg(
        total_revenue = ('price', 'sum')
    )
    .reset_index()
)

print(customer_revenue.shape)

customer_revenue.sort_values(
    'total_revenue',
    ascending=False
).head(10)


(93104, 2)


,customer_unique_id,total_revenue
3716,0a0a92112bd4c708ca5fde585afaa872,13440.0
79420,da122df9eeddfedc1dc1f5349a1a690c,7388.0
43051,763c8b1c9c68a0229c42c9fc6f662b93,7160.0
80246,dc4802a71eae9be1dd28f5d788ceb526,6735.0
25360,459bef486812aa25204be022145caa62,6729.0
92831,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0
23341,4007669dec559734d6f53e029e360987,5934.6
86913,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0
26562,48e1ac109decbb87765a3eade6854098,4590.0
59006,a229eba70ec1c2abef51f04987deb7a5,4400.0


In [43]:
# each customer's total orders
customer_order = (
    customer_sales
    .groupby('customer_unique_id')
    .agg(
        total_orders = ('order_id', 'nunique')
    )
    .reset_index()
)
customer_order.sort_values(
    'total_orders',
    ascending=False
).head(10)


,customer_unique_id,total_orders
51290,8d50f5eadf50201ccdcedfb9e2ac8455,15
22711,3e43e6105506432c953e165fb2acf44c,9
36602,6469f99c1f9dfae7733b25662e7f1782,7
73727,ca77025e7201e3b30c44b472ff346268,7
10035,1b6c7548a2a1f9037c1fd3ddfed95f33,7
80321,dc813062e0fc23409cd255f7f53c7074,6
6947,12f5d6e1cbf93dafd9dcc19095df0b3d,6
26179,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
36397,63cfc61cee11cbe306bff5857d00bfe4,6
87649,f0e310a6839dce9de1638e0fe5ab282a,6


In [44]:
# combine revenue + order
customer_value = customer_revenue.merge(
    customer_order,
    on='customer_unique_id',
    how='inner'
)


In [45]:
# average order value
customer_value['customer_aov'] = (
    customer_value['total_revenue']
    / customer_value['total_orders']
)
customer_value.sort_values(
    'total_revenue',
    ascending=False
).head(10)
customer_value['customer_type'] = np.where(
    customer_value['total_orders'] == 1, 'One-time', 'Repeat'
)
print(customer_value['customer_type'].value_counts())

customer_value.groupby('customer_type').agg(
    customers = ('customer_unique_id', 'count'),
    total_revenue = ('total_revenue', 'sum'),
    avg_revenue = ('total_revenue', 'mean'),
    avg_orders = ('total_orders', 'mean')
).reset_index()


customer_type
One-time    90315
Repeat       2789
Name: count, dtype: int64


,customer_type,customers,total_revenue,avg_revenue,avg_orders
0,One-time,90315,12455672.09,137.913659,1.000000
1,Repeat,2789,725355.04,260.077103,2.114019


##### 90,315 one-time customers
##### 2,789 repeat customers
##### So most customers purchased only once.

In [46]:
# repeat purchase rate
total_customers = customer_value['customer_unique_id'].nunique()

repeat_customers = (
    customer_value[
        customer_value['total_orders'] >= 2
    ]['customer_unique_id']
    .nunique()
)

repeat_purchase_rate = (
    repeat_customers / total_customers
) * 100

print("Total Customers:", total_customers)
print("Repeat Customers:", repeat_customers)
print("Repeat Purchase Rate:", repeat_purchase_rate)


Total Customers: 93104
Repeat Customers: 2789
Repeat Purchase Rate: 2.995574841037979


In [47]:
# revenue share from repeat customers
# How much of our revenue comes from customers who came back?

repeat_revenue = customer_value.loc[
    customer_value['customer_type'] == 'Repeat',
    'total_revenue'
].sum()

total_revenue = customer_value['total_revenue'].sum()

repeat_revenue_share = (
    repeat_revenue / total_revenue
) * 100

print("Repeat Customer Revenue:", repeat_revenue)
print("Total Revenue:", total_revenue)
print("Repeat Customer Revenue Share:", repeat_revenue_share)


Repeat Customer Revenue: 725355.04
Total Revenue: 13181027.129999999
Repeat Customer Revenue Share: 5.503023647899889


##### The observed repeat purchase rate is approximately 3%, indicating substantial opportunity to improve customer retention and repeat purchasing.
### RFM Analysis
Which customers are our best customers, and which customers may need re-engagement?
#### RFM stands for:
#### R — Recency: How recently did the customer purchase?
#### F — Frequency: How many orders did they make?
#### M — Monetary: How much did they spend?

In [48]:
# Recency
# For each customer, when was their most recent purchase?

customer_sales['order_purchase_timestamp'] = pd.to_datetime(
    customer_sales['order_purchase_timestamp']
)
customer_recency = (
    customer_sales
    .groupby('customer_unique_id')
    .agg(
        last_purchase = ('order_purchase_timestamp', 'max')
    )
    .reset_index()
)
print(customer_recency.head())


                 customer_unique_id       last_purchase
0  0000366f3b9a7992bf8c76cfdf3221e2 2018-05-10 10:56:27
1  0000b849f77a49e4a4ce2b2a4ca5be3f 2018-05-07 11:11:27
2  0000f46a3911fa3c0805444483337064 2017-03-10 21:05:03
3  0000f6ccb0745a6a4b88665a16c9f078 2017-10-12 20:29:41
4  0004aac84e0df4da2b147fca70cf8255 2017-11-14 19:45:42


In [49]:
# reference date
reference_date = (
    customer_sales['order_purchase_timestamp'].max()
    + pd.Timedelta(days=1)
)

print(reference_date)

customer_recency['recency'] = (
    reference_date - customer_recency['last_purchase']
).dt.days
customer_recency.head()

customer_recency['recency'].describe()


2018-08-30 15:00:37


count    93104.000000
mean       236.702924
std        150.942219
min          1.000000
25%        114.000000
50%        218.000000
75%        345.000000
max        602.000000
Name: recency, dtype: float64

In [50]:
# Frequency
# How frequently does each customer purchase from the business?

customer_frequency = (
    customer_sales
    .groupby('customer_unique_id')
    .agg(
        frequency=('order_id', 'nunique')
    )
    .reset_index()
)

customer_frequency['frequency'].describe()


count    93104.000000
mean         1.033371
std          0.209012
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.000000
Name: frequency, dtype: float64

In [51]:
# Monetary
# How much revenue does each customer generate?

customer_monetary = (
    customer_sales
    .groupby('customer_unique_id')
    .agg(
        monetary=('price', 'sum')
    )
    .reset_index()
)

customer_monetary.head()
customer_monetary['monetary'].describe()


count    93104.000000
mean       141.573156
std        215.760366
min          0.850000
25%         47.650000
50%         89.700000
75%        154.175000
max      13440.000000
Name: monetary, dtype: float64

### RFM table

In [52]:
rfm = (
    customer_recency
    .merge(
        customer_frequency,
        on='customer_unique_id',
        how='inner'
    )
    .merge(
        customer_monetary,
        on='customer_unique_id',
        how='inner'
    )
)

rfm.head()
rfm.shape


(93104, 5)

In [53]:
# recency scores
rfm['r_score'] = pd.qcut(
    rfm['recency'],
    5,
    labels=[5, 4, 3, 2, 1]
)
rfm[['recency', 'r_score']].head(10)
rfm['r_score'].value_counts().sort_index()


r_score
5    18728
4    18705
3    18709
2    18373
1    18589
Name: count, dtype: int64

In [54]:
# frequency scores
rfm['frequency'].value_counts().sort_index()
rfm['f_score'] = np.select(
    [
        rfm['frequency'] == 1,
        rfm['frequency'] == 2,
        rfm['frequency'] == 3,
        rfm['frequency'].between(4, 5),
        rfm['frequency'] >= 6
    ],
    [1, 2, 3, 4, 5]
)
rfm['f_score'].value_counts().sort_index()


f_score
1    90315
2     2562
3      180
4       37
5       10
Name: count, dtype: int64

In [55]:
# monetary scores
rfm['m_score'] = pd.qcut(
    rfm['monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
)
rfm['m_score'].value_counts().sort_index()
rfm[['monetary', 'm_score']].head(10)


,monetary,m_score
0,129.90,4
1,18.90,1
2,69.00,2
3,25.99,1
4,180.00,5
5,154.00,4
6,27.99,1
7,382.00,5
8,135.00,4
9,104.90,3


### RFM scores

In [56]:
rfm['rfm_score'] = (
    rfm['r_score'].astype(int)
    + rfm['f_score']
    + rfm['m_score'].astype(int)
)
rfm['rfm_score'].describe()
rfm['rfm_score'].value_counts().sort_index()


rfm_score
3      4087
4      7553
5     11423
6     14222
7     18227
8     14570
9     11058
10     7766
11     3808
12      323
13       46
14       17
15        4
Name: count, dtype: int64

### customer segment

In [57]:
rfm['segment'] = np.select(
    [
        (rfm['r_score'] >= 4) &
        (rfm['f_score'] >= 4) &
        (rfm['m_score'] >= 4),

        (rfm['r_score'] >= 4) &
        (rfm['f_score'] <= 2) &
        (rfm['m_score'] >= 4),

        (rfm['r_score'] <= 2) &
        (rfm['f_score'] >= 2) &
        (rfm['m_score'] >= 2),

        (rfm['r_score'] <= 2) &
        (rfm['f_score'] == 1) &
        (rfm['m_score'] <= 2)
    ],
    [
        'Loyal High-Value',
        'Recent High-Value',
        'At-Risk',
        'Lost / Low-Value'
    ],
    default='Regular'
)
rfm['segment'].value_counts()

rfm.groupby('segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).sort_values('customers', ascending=False)


,customers,avg_recency,avg_frequency,avg_monetary
segment,,,,
Regular,32341,253.779908,1.006741,116.700368
Lost / Low-Value,30383,179.631801,1.000000,40.563255
Recent High-Value,29268,281.301968,1.050943,270.500797
At-Risk,1087,123.803128,2.198712,223.300350
Loyal High-Value,25,200.800000,4.800000,585.694000


##### Its average recency is 281 days, which is worse than Regular at 254 days.
##### That's because R >= 4 means "better than the bottom 40% of recency," but R4 doesn't necessarily mean recently purchased in an absolute business sense.
##### Similarly, our Lost / Low-Value group has average recency 180 days, which isn't necessarily "lost."
##### So, the RFM scores are mathematically valid, but our business labels are too aggressive.

In [58]:
rfm['final_segment'] = np.select(
    [
        (rfm['frequency'] >= 3) &
        (rfm['monetary'] >= 300),

        (rfm['recency'] <= 180) &
        (rfm['frequency'] >= 2),

        (rfm['frequency'] == 1) &
        (rfm['monetary'] >= 250),

        (rfm['recency'] > 180) &
        (rfm['frequency'] >= 2)
    ],
    [
        'Loyal Customers',
        'Potential Loyalists',
        'High-Value One-Time',
        'At-Risk Repeat'
    ],
    default='One-Time / Low Engagement'
)

rfm['final_segment'].value_counts()

rfm.groupby('final_segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).sort_values('customers', ascending=False)


,customers,avg_recency,avg_frequency,avg_monetary
final_segment,,,,
One-Time / Low Engagement,80321,236.770297,1.000000,88.142186
High-Value One-Time,9994,240.934761,1.000000,537.923114
At-Risk Repeat,1512,321.800926,2.041667,240.544881
Potential Loyalists,1158,91.223661,2.048359,244.479991
Loyal Customers,119,170.252101,3.672269,660.028151


Loyal Customers — 119
##### These customers average 3.67 orders and ₹660 spend. They're your strongest relationship customers.
##### Focus on retention, loyalty benefits, personalized recommendations.

Potential Loyalists — 1,158
##### They purchased recently (average 91 days) and already have ~2 orders.
##### This is probably one of the most interesting groups to nurture toward stronger loyalty.

At-Risk Repeat — 1,512
##### They have purchased more than once, but their average recency is 322 days.
##### This is a genuine re-engagement opportunity because they have already demonstrated repeat purchasing behavior.

High-Value One-Time — 9,994
##### This is a particularly interesting group: they spend an average ₹538, but have only purchased once.
##### The business question becomes: How do we convert these high-value one-time buyers into repeat customers?

One-Time / Low Engagement — 80,321
##### This is by far the largest group. They average only ₹88 and have made one purchase.

In [59]:
rfm.groupby('final_segment').agg(
    customers=('customer_unique_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_revenue=('monetary', 'mean')
).sort_values('total_revenue', ascending=False)


,customers,total_revenue,avg_revenue
final_segment,,,
One-Time / Low Engagement,80321,7079668.49,88.142186
High-Value One-Time,9994,5376003.60,537.923114
At-Risk Repeat,1512,363703.86,240.544881
Potential Loyalists,1158,283107.83,244.479991
Loyal Customers,119,78543.35,660.028151


In [60]:
# revenue share
segment_revenue = (
    rfm
    .groupby('final_segment')
    .agg(
        customers=('customer_unique_id', 'count'),
        total_revenue=('monetary', 'sum')
    )
    .reset_index()
)

segment_revenue['customer_share_pct'] = (
    segment_revenue['customers']
    / segment_revenue['customers'].sum()
    * 100
)

segment_revenue['revenue_share_pct'] = (
    segment_revenue['total_revenue']
    / segment_revenue['total_revenue'].sum()
    * 100
)

segment_revenue.sort_values(
    'revenue_share_pct',
    ascending=False
)


,final_segment,customers,total_revenue,customer_share_pct,revenue_share_pct
3,One-Time / Low Engagement,80321,7079668.49,86.270192,53.711053
1,High-Value One-Time,9994,5376003.60,10.734233,40.785923
0,At-Risk Repeat,1512,363703.86,1.623990,2.759298
4,Potential Loyalists,1158,283107.83,1.243770,2.147843
2,Loyal Customers,119,78543.35,0.127814,0.595882


High-Value One-Time customers are the standout segment.
##### They represent only 10.73% of customers, but contribute 40.79% of revenue.
##### It means the company has a substantial group of customers who have already demonstrated high spending, but have purchased only once.
Prioritize retention campaigns for high-value one-time customers rather than giving discounts to the entire customer base.
The One-Time / Low Engagement segment contains 86.27% of customers and generates 53.71% of revenue.
##### So we shouldn't simply call this segment "bad customers."
##### It's a large acquisition/engagement population. Because the average monetary value is relatively low, expensive retention campaigns may not be economical for everyone.
A better strategy would be:
##### High-value one-time => high-touch retention
##### One-time/low engagement => low-cost automated engagement
##### At-risk repeat => reactivation
##### Potential loyalists => loyalty building
##### Loyal customers => retention/VIP treatment

#### Do customers actually come back over time, and how well does the business retain each group of newly acquired customers?

In [61]:
# Preparing Customer Purchase Data
customer_sales['order_purchase_timestamp'] = pd.to_datetime(
    customer_sales['order_purchase_timestamp']
)

customer_orders = (
    customer_sales[
        ['customer_unique_id', 'order_id', 'order_purchase_timestamp']
    ]
    .drop_duplicates()
)
customer_orders.shape
customer_orders.head()


,customer_unique_id,order_id,order_purchase_timestamp
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49
3,7c142cf63193a1473d2e66489a9ae977,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06
4,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39


In [62]:
# first purchase
first_purchase = (
    customer_orders
    .groupby('customer_unique_id')
    .agg(
        first_purchase_date=('order_purchase_timestamp', 'min')
    )
    .reset_index()
)
first_purchase.head()


,customer_unique_id,first_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42


In [63]:
# cohort month
first_purchase['cohort_month'] = (
    first_purchase['first_purchase_date']
    .dt.to_period('M')
)
first_purchase.head()
cohort_data = customer_orders.merge(
    first_purchase[
        ['customer_unique_id', 'cohort_month']
    ],
    on='customer_unique_id',
    how='left'
)

cohort_data['purchase_month'] = (
    cohort_data['order_purchase_timestamp']
    .dt.to_period('M')
)

cohort_data.head()
cohort_data['cohort_index'] = (
    (cohort_data['purchase_month'].dt.year -
     cohort_data['cohort_month'].dt.year) * 12
    +
    (cohort_data['purchase_month'].dt.month -
     cohort_data['cohort_month'].dt.month)
)

cohort_data[
    ['customer_unique_id',
     'cohort_month',
     'purchase_month',
     'cohort_index']
].head(10)


,customer_unique_id,cohort_month,purchase_month,cohort_index
0,7c396fd4830fd04220f754e42b4e5bff,2017-09,2017-10,1
1,af07308b275d755c9edb36a90c618231,2018-07,2018-07,0
2,3a653a41f6f9fc3d2a113cf8398680e8,2018-08,2018-08,0
3,7c142cf63193a1473d2e66489a9ae977,2017-11,2017-11,0
4,72632f0f9dd73dfee390c9b22eb56dd6,2018-02,2018-02,0
5,80bb27c7c16e8f973207a5086ab329e2,2017-07,2017-07,0
6,932afa1e708222e5821dac9cd5db4cae,2017-05,2017-05,0
7,39382392765b6dc74812866ee5ee92a7,2017-01,2017-01,0
8,299905e3934e9e181bfb2e164dd4b4f8,2017-07,2017-07,0
9,f2a85dec752b8517b5e58a06ff3cd937,2017-05,2017-05,0


In [64]:
# retained customers
cohort_counts = (
    cohort_data
    .groupby(['cohort_month', 'cohort_index'])
    ['customer_unique_id']
    .nunique()
    .reset_index()
)

cohort_counts = cohort_counts.rename(
    columns={
        'customer_unique_id': 'customers'
    }
)

cohort_counts.head(10)


,cohort_month,cohort_index,customers
0,2017-01,0,718
1,2017-01,1,2
2,2017-01,2,2
3,2017-01,3,1
4,2017-01,4,3
5,2017-01,5,1
6,2017-01,6,3
7,2017-01,7,1
8,2017-01,8,1
9,2017-01,10,3


In [65]:
# cohort retention matrix
retention_matrix = (
    cohort_counts
    .pivot(
        index='cohort_month',
        columns='cohort_index',
        values='customers'
    )
)
retention_matrix.head(10)


cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19
cohort_month,,,,,,,,,,,,,,,,,,,
2017-01,718.0,2.0,2.0,1.0,3.0,1.0,3.0,1.0,1.0,NaN,3.0,1.0,5.0,3.0,1.0,1.0,2.0,3.0,1.0
2017-02,1628.0,3.0,5.0,2.0,7.0,2.0,4.0,3.0,2.0,3.0,2.0,5.0,2.0,3.0,2.0,1.0,1.0,3.0,NaN
2017-03,2503.0,11.0,9.0,10.0,9.0,4.0,4.0,8.0,8.0,2.0,9.0,3.0,5.0,3.0,4.0,6.0,2.0,3.0,NaN
2017-04,2257.0,14.0,5.0,4.0,6.0,6.0,8.0,7.0,7.0,4.0,6.0,2.0,1.0,1.0,2.0,2.0,3.0,NaN,NaN
2017-05,3451.0,16.0,16.0,10.0,10.0,11.0,14.0,5.0,9.0,9.0,9.0,12.0,8.0,1.0,6.0,7.0,NaN,NaN,NaN
2017-06,3037.0,15.0,12.0,13.0,9.0,12.0,11.0,7.0,4.0,6.0,9.0,11.0,5.0,5.0,7.0,NaN,NaN,NaN,NaN
2017-07,3753.0,20.0,13.0,9.0,11.0,8.0,12.0,4.0,7.0,10.0,8.0,11.0,5.0,9.0,NaN,NaN,NaN,NaN,NaN
2017-08,4057.0,28.0,14.0,11.0,14.0,21.0,12.0,11.0,6.0,6.0,10.0,8.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN
2017-09,4005.0,28.0,22.0,11.0,18.0,9.0,9.0,10.0,11.0,8.0,10.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
# Convert to retention %
retention_pct = (
    retention_matrix
    .div(retention_matrix[0], axis=0)
    * 100
)
retention_pct.round(2).head(10)


cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19
cohort_month,,,,,,,,,,,,,,,,,,,
2017-01,100.0,0.28,0.28,0.14,0.42,0.14,0.42,0.14,0.14,NaN,0.42,0.14,0.70,0.42,0.14,0.14,0.28,0.42,0.14
2017-02,100.0,0.18,0.31,0.12,0.43,0.12,0.25,0.18,0.12,0.18,0.12,0.31,0.12,0.18,0.12,0.06,0.06,0.18,NaN
2017-03,100.0,0.44,0.36,0.40,0.36,0.16,0.16,0.32,0.32,0.08,0.36,0.12,0.20,0.12,0.16,0.24,0.08,0.12,NaN
2017-04,100.0,0.62,0.22,0.18,0.27,0.27,0.35,0.31,0.31,0.18,0.27,0.09,0.04,0.04,0.09,0.09,0.13,NaN,NaN
2017-05,100.0,0.46,0.46,0.29,0.29,0.32,0.41,0.14,0.26,0.26,0.26,0.35,0.23,0.03,0.17,0.20,NaN,NaN,NaN
2017-06,100.0,0.49,0.40,0.43,0.30,0.40,0.36,0.23,0.13,0.20,0.30,0.36,0.16,0.16,0.23,NaN,NaN,NaN,NaN
2017-07,100.0,0.53,0.35,0.24,0.29,0.21,0.32,0.11,0.19,0.27,0.21,0.29,0.13,0.24,NaN,NaN,NaN,NaN,NaN
2017-08,100.0,0.69,0.35,0.27,0.35,0.52,0.30,0.27,0.15,0.15,0.25,0.20,0.12,NaN,NaN,NaN,NaN,NaN,NaN
2017-09,100.0,0.70,0.55,0.27,0.45,0.22,0.22,0.25,0.27,0.20,0.25,0.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Customer retention is weak, with monthly cohort retention generally remaining below 1% in the observed early cohorts. This is consistent with the low observed repeat-purchase rate and suggests a significant opportunity to improve post-purchase engagement and repeat purchasing

In [67]:
mature_cohorts = retention_pct[
    retention_pct[6].notna()
]

mature_cohorts[[0, 1, 2, 3, 4, 5, 6]].mean().round(2)


cohort_index
0    100.00
1      0.47
2      0.35
3      0.25
4      0.31
5      0.24
6      0.26
dtype: float64

##### Only about 0.47% of customers, on average, made another purchase in the month immediately following their first purchase.
##### After that, retention stays extremely low, generally around 0.2–0.4%.
##### This aligns with our earlier finding of an approximately 3% overall repeat-purchase rate.

In [68]:
print(df_or.shape)
print(df_or.head())


(99224, 7)
                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_score review_comment_title  \
0             4                  NaN   
1             5                  NaN   
2             5                  NaN   
3             5                  NaN   
4             5                  NaN   

                              review_comment_message review_creation_date  \
0                                                NaN  2018-01-18 00:00:00   
1                                                NaN  2018-03-10 00:00:00   
2                                                NaN  2018-02-17 00:00:00   
3

#### We want to understand whether delivery performance is associated with customer satisfaction.

1. How long does delivery usually take?
2. How often are orders delivered late?
3. Are late orders associated with lower review scores?
4. Which categories have the biggest satisfaction problems?
#### Does the delivery experience appear to influence customer satisfaction?

In [69]:
# delivery analysis
delivery_analysis = df_o[
    [
        'order_id',
        'order_status',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
].copy()
date_columns = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    delivery_analysis[col] = pd.to_datetime(
        delivery_analysis[col]
    )


In [70]:
# delivery date
delivery_analysis['delivery_days'] = (
    delivery_analysis['order_delivered_customer_date']
    - delivery_analysis['order_purchase_timestamp']
).dt.total_seconds() / (24 * 60 * 60)
delivery_analysis['delivery_days'].describe()


count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_days, dtype: float64

In [71]:
# late
delivery_analysis['is_late'] = (
    delivery_analysis['order_delivered_customer_date']
    > delivery_analysis['order_estimated_delivery_date']
)
delivery_analysis['is_late'].value_counts(
    normalize=True
) * 100


is_late
False    92.129001
True      7.870999
Name: proportion, dtype: float64

In [72]:
# review scores
delivery_reviews = delivery_analysis.merge(
    df_or[
        ['order_id', 'review_score']
    ],
    on='order_id',
    how='left'
)
delivery_reviews.shape
delivery_reviews.head()

delivery_reviews['review_score'].value_counts(
    normalize=True
).sort_index() * 100
delivery_reviews['review_score'].mean()

np.float64(4.08642062404257)

In [73]:
# late vs on-time
delivery_reviews.groupby('is_late').agg(
    orders=('order_id', 'nunique'),
    avg_review_score=('review_score', 'mean')
)


,orders,avg_review_score
is_late,,
False,91614,4.214307
True,7827,2.566550


In [74]:
# low-rating rate
delivery_reviews['low_rating'] = (
    delivery_reviews['review_score'] <= 2
)
delivery_reviews.groupby('is_late').agg(
    orders=('order_id', 'nunique'),
    avg_review_score=('review_score', 'mean'),
    low_rating_rate=('low_rating', 'mean')
)
satisfaction_by_delivery = (
    delivery_reviews
    .groupby('is_late')
    .agg(
        orders=('order_id', 'nunique'),
        avg_review_score=('review_score', 'mean'),
        low_rating_rate=('low_rating', 'mean')
    )
    .reset_index()
)

satisfaction_by_delivery['low_rating_rate'] *= 100

satisfaction_by_delivery
delivery_reviews['delay_days'] = (
    delivery_reviews['order_delivered_customer_date']
    - delivery_reviews['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 60 * 60)

delivery_reviews['delay_category'] = np.select(
    [
        delivery_reviews['delay_days'] <= 0,
        delivery_reviews['delay_days'].between(1, 3),
        delivery_reviews['delay_days'].between(4, 7),
        delivery_reviews['delay_days'] > 7
    ],
    [
        'On Time',
        '1-3 Days Late',
        '4-7 Days Late',
        '8+ Days Late'
    ],
    default='Unknown'
)
delay_satisfaction = (
    delivery_reviews
    .groupby('delay_category')
    .agg(
        orders=('order_id', 'nunique'),
        avg_review_score=('review_score', 'mean'),
        low_rating_rate=('low_rating', 'mean')
    )
    .reset_index()
)

delay_satisfaction['low_rating_rate'] *= 100

delay_satisfaction


,delay_category,orders,avg_review_score,low_rating_rate
0,1-3 Days Late,1370,3.511029,25.254731
1,4-7 Days Late,1319,2.174083,63.492063
2,8+ Days Late,3346,1.731439,76.143791
3,On Time,88649,4.293578,9.187898
4,Unknown,4757,2.490112,55.010438


##### As delivery delays become more severe, customer satisfaction declines substantially.

### Category-Level Satisfaction
Which product categories have the biggest customer-satisfaction problems?

In [75]:
order_category = (
    df_oi[['order_id', 'product_id']]
    .merge(
        df_p[['product_id', 'product_category_name']],
        on='product_id',
        how='left'
    )
)

category_satisfaction = delivery_reviews.merge(
    order_category,
    on='order_id',
    how='left'
)

category_review = (
    category_satisfaction
    .groupby('product_category_name')
    .agg(
        orders = ('order_id', 'nunique'),
        avg_review_score = ('review_score', 'mean'),
        low_rating_rate = ('low_rating', 'mean')
    )
    .reset_index()
)

category_review['low_rating_rate'] *= 100


In [76]:
# Highest-volume categories
category_review.sort_values(
    'orders',
    ascending=False
).head(10)


,product_category_name,orders,avg_review_score,low_rating_rate
13,cama_mesa_banho,9417,3.895663,18.740018
11,beleza_saude,8836,4.142768,13.580755
32,esporte_lazer,7720,4.107986,14.482759
44,informatica_acessorios,6689,3.930819,18.507727
54,moveis_decoracao,6449,3.903493,19.263220
72,utilidades_domesticas,5884,4.055019,15.538704
66,relogios_presentes,5624,4.019160,16.113981
70,telefonia,4199,3.946867,16.725275
8,automotivo,3897,4.065512,15.108083
12,brinquedos,3886,4.158641,13.381818


In [77]:
# Worst-rated categories
category_review.sort_values(
    'avg_review_score',
    ascending=True
).head(10)


,product_category_name,orders,avg_review_score,low_rating_rate
67,seguros_e_servicos,2,2.500000,50.000000
42,fraldas_higiene,27,3.256410,25.641026
65,portateis_cozinha_e_preparadores_de_alimentos,14,3.266667,33.333333
60,pc_gamer,8,3.333333,33.333333
55,moveis_escritorio,1273,3.493183,25.867137
15,casa_conforto_2,24,3.629630,23.333333
38,fashion_roupa_masculina,112,3.641221,28.030303
71,telefonia_fixa,217,3.683206,25.283019
5,artigos_de_festas,39,3.767442,23.255814
36,fashion_roupa_feminina,39,3.780000,18.000000


In [78]:
# delivery performance by category
category_delivery = (
    category_satisfaction
    .groupby('product_category_name')
    .agg(
        orders = ('order_id', 'nunique'),
        avg_delivery_days = ('delivery_days', 'mean'),
        late_rate = ('is_late', 'mean')
    )
    .reset_index()
)
category_delivery['late_rate'] *= 100
category_delivery.sort_values(
    'orders',
    ascending=False
).head(10)
category_delivery.sort_values(
    'late_rate',
    ascending=False
).head(10)


,product_category_name,orders,avg_delivery_days,late_rate
15,casa_conforto_2,24,14.573525,16.666667
52,moveis_colchao_e_estofado,38,14.432970,13.157895
7,audio,350,13.328610,12.602740
39,fashion_underwear_e_moda_praia,121,13.706326,12.213740
6,artigos_de_natal,128,15.740684,11.764706
49,livros_tecnicos,260,10.654750,10.780669
14,casa_conforto,397,13.522274,10.068650
23,construcao_ferramentas_iluminacao,244,9.702170,9.836066
30,eletronicos,2550,12.887729,9.609827
1,alimentos,450,9.559812,9.607843


In [79]:
# satisfaction + delivery
category_performance = category_review.merge(
    category_delivery[
        ['product_category_name', 'avg_delivery_days', 'late_rate']
    ],
    on='product_category_name',
    how='inner'
)

category_performance_meaningful = category_performance[
    category_performance['orders'] >= 500
].copy()


In [80]:
# lowest satisfaction
category_performance_meaningful.sort_values(
    'avg_review_score',
    ascending=True
).head(10)


,product_category_name,orders,avg_review_score,low_rating_rate,avg_delivery_days,late_rate
55,moveis_escritorio,1273,3.493183,25.867137,20.854464,8.818342
13,cama_mesa_banho,9417,3.895663,18.740018,12.813550,8.278616
54,moveis_decoracao,6449,3.903493,19.263220,12.875544,8.199643
44,informatica_acessorios,6689,3.930819,18.507727,13.240704,7.562706
70,telefonia,4199,3.946867,16.725275,12.863408,8.109890
9,bebes,2885,4.011811,16.563619,12.522364,8.525870
66,relogios_presentes,5624,4.019160,16.113981,12.662845,8.098650
20,consoles_games,1062,4.023070,15.202109,13.611720,7.469244
30,eletronicos,2550,4.037468,15.643064,12.887729,9.609827
40,ferramentas_jardim,3518,4.042735,15.959642,13.703470,7.842238


In [81]:
# highest late rate
category_performance_meaningful.sort_values(
    'late_rate',
    ascending=False
).head(10)


,product_category_name,orders,avg_review_score,low_rating_rate,avg_delivery_days,late_rate
30,eletronicos,2550,4.037468,15.643064,12.887729,9.609827
11,beleza_saude,8836,4.142768,13.580755,11.987234,8.872211
55,moveis_escritorio,1273,3.493183,25.867137,20.854464,8.818342
9,bebes,2885,4.011811,16.563619,12.522364,8.525870
13,cama_mesa_banho,9417,3.895663,18.740018,12.813550,8.278616
45,instrumentos_musicais,628,4.152593,13.950073,12.972337,8.223201
54,moveis_decoracao,6449,3.903493,19.263220,12.875544,8.199643
70,telefonia,4199,3.946867,16.725275,12.863408,8.109890
66,relogios_presentes,5624,4.019160,16.113981,12.662845,8.098650
8,automotivo,3897,4.065512,15.108083,12.227972,8.082707


##### Delivery experience is strongly associated with customer satisfaction overall, with severe delays producing substantially higher low-rating rates. At the category level, moveis_escritorio stands out with the lowest average rating among meaningful-volume categories and the longest average delivery time, although its late-delivery rate is only moderately above the overall rate. This suggests that delivery performance may contribute to dissatisfaction but is unlikely to be the only factor

### Payment & Order Value Analysis
How are customers paying, and does payment method appear to be associated with order value?

In [82]:
df_op.head()
payment_method_summary = (
    df_op
    .groupby('payment_type')
    .agg(
        payments = ('order_id', 'count'),
        unique_orders = ('order_id', 'nunique'),
        total_payment = ('payment_value', 'sum'),
        avg_payment = ('payment_value', 'mean')
    )
    .reset_index()
    .sort_values('unique_orders', ascending=False)
)

payment_method_summary


,payment_type,payments,unique_orders,total_payment,avg_payment
1,credit_card,76795,76505,12542084.19,163.319021
0,boleto,19784,19784,2869361.27,145.034435
4,voucher,5775,3866,379436.87,65.703354
2,debit_card,1529,1528,217989.79,142.570170
3,not_defined,3,3,0.00,0.000000


In [83]:
# Payment-method-associated order share
payment_method_summary['order_share_pct'] = (
    payment_method_summary['unique_orders']
    / payment_method_summary['unique_orders'].sum()
    * 100
)
payment_method_summary


,payment_type,payments,unique_orders,total_payment,avg_payment,order_share_pct
1,credit_card,76795,76505,12542084.19,163.319021,75.236512
0,boleto,19784,19784,2869361.27,145.034435,19.455972
4,voucher,5775,3866,379436.87,65.703354,3.801900
2,debit_card,1529,1528,217989.79,142.570170,1.502665
3,not_defined,3,3,0.00,0.000000,0.002950


##### Credit card is overwhelmingly the dominant payment method, appearing in about 75% of payment-associated orders.
##### Boleto is the second-largest at about 19.5%.
##### Voucher has a much lower average payment value, which makes sense because vouchers can be used alongside other payment methods.

### installments
How important are installment payments, particularly for credit-card purchases?

In [84]:
installment_summary = (
    df_op[df_op['payment_type'] == 'credit_card']
    .groupby('payment_installments')
    .agg(
        payments=('order_id', 'count'),
        unique_orders=('order_id', 'nunique'),
        total_payment=('payment_value', 'sum'),
        avg_payment=('payment_value', 'mean')
    )
    .reset_index()
    .sort_values('payment_installments')
)

installment_summary
installment_summary['order_share_pct'] = (
    installment_summary['unique_orders']
    / installment_summary['unique_orders'].sum()
    * 100
)

installment_summary
credit_card_payments = df_op[
    df_op['payment_type'] == 'credit_card'
].copy()

credit_card_payments['installment_band'] = np.select(
    [
        credit_card_payments['payment_installments'] == 1,
        credit_card_payments['payment_installments'].between(2, 3),
        credit_card_payments['payment_installments'].between(4, 6),
        credit_card_payments['payment_installments'].between(7, 10),
        credit_card_payments['payment_installments'] > 10
    ],
    [
        '1 Installment',
        '2-3 Installments',
        '4-6 Installments',
        '7-10 Installments',
        '11+ Installments'
    ],
    default='Other'
)
installment_band_summary = (
    credit_card_payments
    .groupby('installment_band')
    .agg(
        unique_orders=('order_id', 'nunique'),
        total_payment=('payment_value', 'sum'),
        avg_payment=('payment_value', 'mean'),
        median_payment=('payment_value', 'median')
    )
    .reset_index()
)

installment_band_summary


,installment_band,unique_orders,total_payment,avg_payment,median_payment
0,1 Installment,25407,2440445.43,95.872930,67.870
1,11+ Installments,341,122195.59,358.344839,214.590
2,2-3 Installments,22822,3070386.83,134.230429,109.725
3,4-6 Installments,16223,2947693.72,181.318430,127.510
4,7-10 Installments,11832,3961173.99,333.825551,205.835
5,Other,2,188.63,94.315000,94.315


##### Credit-card installment usage is strongly associated with order/payment value: customers using longer installment plans tend to make higher-value purchases. This suggests installment flexibility may be particularly relevant for higher-ticket purchases.

In [85]:
credit_card_orders = (
    df_op[df_op['payment_type'] == 'credit_card']
    .groupby('order_id')
    .agg(
        installments=('payment_installments', 'max')
    )
    .reset_index()
)

multi_installment_share = (
    (credit_card_orders['installments'] > 1).mean() * 100
)

multi_installment_share
credit_card_orders['installment_type'] = np.where(
    credit_card_orders['installments'] == 1,
    'Single Payment',
    'Multiple Installments'
)

credit_card_orders['installment_type'].value_counts(normalize=True) * 100


installment_type
Multiple Installments    66.887132
Single Payment           33.112868
Name: proportion, dtype: float64

##### 66.88% of credit-card orders use multiple installments.
##### 33.11% are single-payment transactions.
##### Longer installment plans are associated with higher payment values.
##### Credit card is the dominant payment method, associated with 75.24% of payment-associated orders.
##### Boleto is the second-largest at 19.46%.

##### Credit cards dominate e-commerce payments, with roughly two-thirds of credit-card orders using multiple installments. Higher-value payments are associated with longer installment plans, highlighting installment flexibility as an important feature for higher-ticket purchases.

### Seller Performance
Which sellers are generating strong sales, and are high-sales sellers also delivering a good customer experience?

In [86]:
seller_performance = (
    sales
    .groupby('seller_id')
    .agg(
        orders=('order_id', 'nunique'),
        items_sold=('order_item_id', 'count'),
        revenue=('price', 'sum')
    )
    .reset_index()
)


In [87]:
# AOV
seller_performance['aov'] = (
    seller_performance['revenue']
    / seller_performance['orders']
)


In [88]:
# best sellers by revenue
seller_performance.sort_values(
    'revenue',
    ascending=False
).head(10)


,seller_id,orders,items_sold,revenue,aov
829,4869f7a5dfa277a7dca6462dcf3b52b2,1124,1148,226987.93,201.946557
975,53243585a1d6dc2643021fd1853d8905,348,400,217940.44,626.265632
853,4a3ca9315b744ce9f8e9374361493884,1772,1949,196882.12,111.107291
2880,fa1c13f2614d7b5c4749cbc52fecda94,578,579,190917.14,330.306471
1472,7c67e1448b00f6e969d365cea6b010ab,973,1355,186570.05,191.747225
1496,7e93a43ef30c4f03f38b393420bc753a,318,321,164582.49,517.555000
2524,da8622b14eb17ae2831f4ac5b9dab84a,1311,1548,159816.87,121.904554
1442,7a67c85e85bb2ce8582c35f2203ad736,1145,1155,139658.69,121.972655
186,1025f0e2d44d7041d6cf58b6550e0bfa,910,1420,138208.56,151.877538
1747,955fee9216a65b617aa5c0531780ce60,1261,1472,131836.71,104.549334


In [89]:
# sellers by order volume
seller_performance.sort_values(
    'orders',
    ascending=False
).head(10)
seller_experience = delivery_reviews.merge(
    df_oi[['order_id', 'seller_id']],
    on='order_id',
    how='left'
)
seller_experience_summary = (
    seller_experience
    .groupby('seller_id')
    .agg(
        orders=('order_id', 'nunique'),
        avg_delivery_days=('delivery_days', 'mean'),
        late_rate=('is_late', 'mean'),
        avg_review_score=('review_score', 'mean'),
        low_rating_rate=('low_rating', 'mean')
    )
    .reset_index()
)

seller_experience_summary['late_rate'] *= 100
seller_experience_summary['low_rating_rate'] *= 100
seller_final = seller_performance.merge(
    seller_experience_summary,
    on='seller_id',
    how='left'
)


In [90]:
# top 10 sellers by revenue
seller_final.sort_values(
    'revenue',
    ascending=False
).head(10)
seller_experience = delivery_reviews[
    (delivery_reviews['order_status'] == 'delivered') &
    (pd.to_datetime(
        delivery_reviews['order_purchase_timestamp']
    ).dt.year >= 2017)
].merge(
    df_oi[['order_id', 'seller_id']],
    on='order_id',
    how='left'
)

seller_experience_summary = (
    seller_experience
    .groupby('seller_id')
    .agg(
        orders=('order_id', 'nunique'),
        avg_delivery_days=('delivery_days', 'mean'),
        late_rate=('is_late', 'mean'),
        avg_review_score=('review_score', 'mean'),
        low_rating_rate=('low_rating', 'mean')
    )
    .reset_index()
)

seller_experience_summary['late_rate'] *= 100
seller_experience_summary['low_rating_rate'] *= 100

seller_final = seller_performance.merge(
    seller_experience_summary,
    on='seller_id',
    how='left'
)


In [91]:
# final check
seller_final.sort_values(
    'revenue',
    ascending=False
).head(10)
seller_final['seller_risk'] = np.where(
    (seller_final['revenue'] >= 100000) &
    (seller_final['avg_review_score'] < 4.0),
    'High Risk',
    'Normal'
)
seller_final[
    seller_final['seller_risk'] == 'High Risk'
].sort_values(
    'revenue',
    ascending=False
)[
    [
        'seller_id',
        'revenue',
        'orders_x',
        'aov',
        'avg_delivery_days',
        'late_rate',
        'avg_review_score',
        'low_rating_rate',
        'seller_risk'
    ]
]


,seller_id,revenue,orders_x,aov,avg_delivery_days,late_rate,avg_review_score,low_rating_rate,seller_risk
853,4a3ca9315b744ce9f8e9374361493884,196882.12,1772,111.107291,14.403202,10.958904,3.828630,18.873668,High Risk
1472,7c67e1448b00f6e969d365cea6b010ab,186570.05,973,191.747225,22.391664,9.590044,3.354197,29.062958,High Risk
186,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56,910,151.877538,12.016974,9.135286,3.867792,19.874477,High Risk
1182,6560211a19b47992c3666cc44a7e94c0,120702.83,1819,66.356696,9.548818,6.243756,3.951637,16.533467,High Risk
355,1f50f920176fa81dab994f9023523100,106655.71,1399,76.237105,15.570992,9.509044,3.987545,17.984496,High Risk


##### Seller performance varies considerably when revenue and customer experience are evaluated together. Some high-revenue sellers maintain strong ratings and delivery performance, while others combine substantial sales with relatively poor customer satisfaction. Seller 1472 stands out as a priority for investigation because it generated high revenue but had the lowest average rating, highest low-rating rate, and longest average delivery time among the high-revenue sellers analyzed.
### Geographic Analysis
#### Which regions generate the most business, and are there geographic differences in delivery performance?

In [92]:
state_sales = (
    customer_sales
    .groupby('customer_state')
    .agg(
        customers=('customer_unique_id', 'nunique'),
        orders=('order_id', 'nunique'),
        revenue=('price', 'sum')
    )
    .reset_index()
)

state_sales['aov'] = (
    state_sales['revenue']
    / state_sales['orders']
)


In [93]:
# top states by revenue
state_sales.sort_values(
    'revenue',
    ascending=False
).head(10)


,customer_state,customers,orders,revenue,aov
25,SP,39065,40406,5055587.13,125.119713
18,RJ,11879,12310,1751433.85,142.277323
10,MG,10969,11319,1548206.88,136.779475
22,RS,5151,5328,726373.92,136.331441
17,PR,4750,4903,664048.00,135.437079
23,SC,3441,3537,504774.07,142.712488
4,BA,3155,3253,493339.26,151.656705
6,DF,2013,2074,295454.64,142.456432
8,GO,1889,1950,281932.21,144.580621
7,ES,1925,1992,267783.66,134.429548


In [94]:
# top states by orders
state_sales.sort_values(
    'orders',
    ascending=False
).head(10)
delivery_reviews


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_late,review_score,low_rating,delay_days,delay_category
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,False,4.0,False,-7.107488,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,False,4.0,False,-5.355729,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,False,5.0,False,-17.245498,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.208750,False,5.0,False,-12.980069,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.873877,False,5.0,False,-9.238171,On Time
...,...,...,...,...,...,...,...,...,...,...,...
99987,9c5dedf39a927c1b2549525ed64a053c,delivered,2017-03-09 09:54:05,2017-03-17 15:08:01,2017-03-28,8.218009,False,5.0,False,-10.369433,On Time
99988,63943bddc261676b46f01ca7ac2f7bd8,delivered,2018-02-06 12:58:58,2018-02-28 17:37:56,2018-03-02,22.193727,False,4.0,False,-1.265324,On Time
99989,83c1379a015df1e13d02aae0204711ab,delivered,2017-08-27 14:46:43,2017-09-21 11:24:17,2017-09-27,24.859421,False,5.0,False,-5.524803,On Time
99990,11c177c8e97725db2631073c19f07b62,delivered,2018-01-08 21:28:27,2018-01-25 23:32:54,2018-02-15,17.086424,False,2.0,True,-20.018819,On Time


### Geographic delivery performance
Do customers in different states experience different delivery performance?

In [95]:
state_delivery = delivery_reviews.merge(
    df_o[['order_id', 'customer_id']],
    on='order_id',
    how='left'
).merge(
    df_c[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

state_delivery_summary = (
    state_delivery
    .groupby('customer_state')
    .agg(
        orders=('order_id', 'nunique'),
        avg_delivery_days=('delivery_days', 'mean'),
        late_rate=('is_late', 'mean'),
        avg_review_score=('review_score', 'mean'),
        low_rating_rate=('low_rating', 'mean')
    )
    .reset_index()
)

state_delivery_summary['late_rate'] *= 100
state_delivery_summary['low_rating_rate'] *= 100
state_delivery_meaningful = state_delivery_summary[
    state_delivery_summary['orders'] >= 500
].copy()
state_delivery_meaningful.sort_values(
    'late_rate',
    ascending=False
).head(10)
state_delivery_meaningful.sort_values(
    'avg_review_score'
).head(10)


,customer_state,orders,avg_delivery_days,late_rate,avg_review_score,low_rating_rate
9,MA,747,21.521970,18.774967,3.764075,21.571238
13,PA,975,23.776525,11.926606,3.849174,19.367992
5,CE,1336,21.268040,14.712472,3.851016,19.716206
4,BA,3380,19.356057,13.570798,3.860888,18.663527
18,RJ,12852,15.319487,12.946636,3.874971,20.471771
15,PE,1652,18.418384,10.342754,4.011543,16.476248
14,PB,536,20.403993,10.614525,4.018832,16.387337
7,ES,2033,15.798111,11.992168,4.041667,14.684288
8,GO,2020,15.604079,7.952872,4.042490,14.727541
6,DF,2140,12.996125,6.898148,4.064711,14.953704


### Business Performance Cross-Analysis
Which product categories combine strong revenue with strong profitability?


### Category Revenue vs. Profit

In [96]:
category_profitability = (
    gross_profit_per_product
    .merge(
        df_p[['product_id']],
        on='product_id',
        how='left'
    )
    .groupby('product_category_name')
    .agg(
        revenue=('price', 'sum'),
        gross_profit=('gross_profit', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)

category_profitability['gross_margin_pct'] = (
    category_profitability['gross_profit']
    / category_profitability['revenue']
    * 100
)
category_profitability_meaningful = (
    category_profitability[
        category_profitability['orders'] >= 500
    ]
    .copy()
)


In [97]:
# Top categories by gross profit
category_profitability_meaningful.sort_values(
    'gross_profit',
    ascending=False
).head(10)


,product_category_name,revenue,gross_profit,orders,gross_margin_pct
11,beleza_saude,1229557.50,382279.8555,8610,31.090848
13,cama_mesa_banho,1022955.77,343989.2500,9267,33.626992
66,relogios_presentes,1163465.91,341272.6325,5491,29.332414
32,esporte_lazer,952840.40,306343.5860,7513,32.150566
44,informatica_acessorios,888055.59,285210.9610,6518,32.116341
54,moveis_decoracao,706237.17,237470.9870,6258,33.624821
72,utilidades_domesticas,614341.62,202958.7185,5734,33.036785
26,cool_stuff,609158.00,186546.9510,3552,30.623738
8,automotivo,577838.39,177842.6030,3802,30.777222
12,brinquedos,467179.98,150372.1660,3782,32.187202


In [98]:
# Top categories by gross margin
category_profitability_meaningful.sort_values(
    'gross_margin_pct',
    ascending=False
).head(10)


,product_category_name,revenue,gross_profit,orders,gross_margin_pct
33,fashion_bolsas_e_acessorios,148945.19,50953.6355,1812,34.209655
59,papelaria,223788.69,75336.3240,2264,33.664044
13,cama_mesa_banho,1022955.77,343989.2500,9267,33.626992
54,moveis_decoracao,706237.17,237470.9870,6258,33.624821
30,eletronicos,154935.94,51775.9660,2516,33.417660
72,utilidades_domesticas,614341.62,202958.7185,5734,33.036785
70,telefonia,309631.94,101098.5965,4086,32.651217
63,pet_shop,211005.96,68749.9675,1684,32.582003
62,perfumaria,385315.55,124999.0015,3060,32.440684
50,malas_acessorios,138701.63,44974.6575,1019,32.425471


##### Category performance looks different when we consider revenue, total estimated gross profit, and margin together. Beleza Saúde generated the highest estimated gross profit, while Cama Mesa Banho combined high order volume with strong revenue and an estimated gross margin above 33%. The analysis also shows that the category with the highest margin does not necessarily generate the highest total profit, highlighting the importance of evaluating both profitability rate and business scale.

## Key Business Findings

### Sales
- November 2017 generated the highest monthly revenue in the reliable analysis period.
- February 2017 showed the strongest meaningful MoM revenue growth.

### Products & Categories
- A small number of categories contribute a large share of sales.
- Order volume and revenue do not always tell the same story.

### Profitability
- Cama Mesa Banho combines high order volume, strong revenue, and a strong estimated margin.
- Profitability figures are based on synthetic product-cost estimates.

### Customers
- Most customers made only one purchase.
- Repeat customers represent a small share of customers.
- High-Value One-Time customers are an important retention opportunity.

### Retention
- Monthly cohort retention is very low.
- The data suggests a significant opportunity to improve repeat purchasing.

### Delivery & Satisfaction
- Late deliveries are strongly associated with lower review scores.
- Severe delays have a particularly high low-rating rate.
- Moveis Escritorio stands out among meaningful-volume categories for weaker satisfaction and longer delivery times.

### Sellers
- Some high-revenue sellers have strong customer experience, while others need attention.
- Seller 1472 is a priority for investigation based on the defined risk criteria.

### Geography
- São Paulo is the largest market by revenue and orders.
- Delivery performance varies considerably across states.

### Payments
- Credit cards are the dominant payment method.
- Longer installment plans are associated with higher payment values.

## Transition to SQL

The Python analysis helped us understand the business questions and identify important patterns.

In the next stage, will reproduce the most important business questions using MySQL to demonstrate SQL-based business analysis.